# ATS Resume Compliance Checker — Fine-Tuning Pipeline
Complete end-to-end pipeline: environment setup, dataset preparation, model loading,
LoRA fine-tuning, inference testing, and evaluation metrics.

**Instructions:** Upload the entire `colab/` folder to Google Drive (e.g., `My Drive/colab/`),
then run cells sequentially from top to bottom.

---
## 1. Environment Setup

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'  # Adjust if you uploaded elsewhere
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies (Colab already has PyTorch + CUDA)
!pip install -q transformers>=4.36.0 peft>=0.7.0 datasets>=2.16.0 accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0 scipy sentencepiece protobuf
!pip install -q pyyaml tqdm matplotlib seaborn

In [ ]:
# Verify installations and GPU
import torch
import transformers
import peft
import datasets
import accelerate

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    if gpu_mem >= 16:
        print("[OK] Sufficient GPU memory for Phi-3 Mini with 4-bit quantization")
    elif gpu_mem >= 8:
        print("[WARNING] Limited GPU memory - 4-bit quantization required")
    else:
        print("[WARNING] Low GPU memory - consider a higher-tier Colab runtime")
else:
    print("[WARNING] No GPU detected - go to Runtime > Change runtime type > GPU")

try:
    import bitsandbytes as bnb
    print(f"\nbitsandbytes: {bnb.__version__} [OK]")
except ImportError:
    print("\n[ERROR] bitsandbytes not installed: !pip install bitsandbytes")

In [ ]:
# Verify folder structure
from pathlib import Path

for item in ["data", "configs", "data/raw_dataset.json",
             "configs/training_config.yaml", "configs/lora_config.yaml"]:
    status = "[OK]" if Path(item).exists() else "[MISSING]"
    print(f"  {status} {item}")

print("\nEnvironment setup complete!")

---
## 2. Dataset Preparation

In [ ]:
import json
import random
from collections import Counter

import yaml

# Load configuration
with open("configs/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

RAW_PATH = config["raw_dataset"]
TRAIN_PATH = config["train_dataset"]
VAL_PATH = config["validation_dataset"]
TRAIN_SPLIT = config["train_split"]
SEED = config["seed"]

print(f"Raw dataset: {RAW_PATH}")
print(f"Train split: {TRAIN_SPLIT}")
print(f"Seed: {SEED}")

In [ ]:
# Load and validate raw dataset
with open(RAW_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data)} samples")
print(f"Sample keys: {list(raw_data[0].keys())}")

def validate_sample(sample):
    required_keys = {"instruction", "input", "output"}
    if not required_keys.issubset(sample.keys()):
        return False, "Missing required keys"
    try:
        output = json.loads(sample["output"])
    except (json.JSONDecodeError, TypeError):
        return False, "Invalid JSON in output"
    required_output_keys = {
        "ats_score", "score_breakdown", "matched_skills",
        "missing_skills", "weak_bullets", "formatting_issues",
        "overall_feedback"
    }
    if not required_output_keys.issubset(output.keys()):
        return False, f"Missing output keys: {required_output_keys - output.keys()}"
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False, "Invalid ATS score"
    return True, "Valid"

valid_samples = []
invalid_samples = []
for i, sample in enumerate(raw_data):
    is_valid, reason = validate_sample(sample)
    if is_valid:
        valid_samples.append(sample)
    else:
        invalid_samples.append((i, reason))

print(f"Valid: {len(valid_samples)} | Invalid: {len(invalid_samples)}")
for idx, reason in invalid_samples[:5]:
    print(f"  Sample {idx}: {reason}")

In [ ]:
# Score distribution
scores = [json.loads(s["output"])["ats_score"] for s in valid_samples]
print(f"ATS Scores — Min: {min(scores)}, Max: {max(scores)}, Mean: {sum(scores)/len(scores):.1f}")

buckets = Counter((s // 10) * 10 for s in scores)
for bucket in sorted(buckets):
    print(f"  {bucket:3d}-{bucket+9:3d}: {'#' * buckets[bucket]} ({buckets[bucket]})")

In [ ]:
# Format prompts, split, and save
def format_prompt(sample, eos_token="<|endoftext|>"):
    return (
        f"### Instruction:\n{sample['instruction']}\n\n"
        f"### Input:\n{sample['input']}\n\n"
        f"### Response:\n{sample['output']}{eos_token}"
    )

for sample in valid_samples:
    sample["text"] = format_prompt(sample)

lengths = [len(s["text"]) for s in valid_samples]
print(f"Prompt char lengths — Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.0f}")

random.seed(SEED)
random.shuffle(valid_samples)
split_idx = int(len(valid_samples) * TRAIN_SPLIT)
train_data = valid_samples[:split_idx]
val_data = valid_samples[split_idx:]
print(f"Train: {len(train_data)} | Validation: {len(val_data)}")

os.makedirs(os.path.dirname(TRAIN_PATH), exist_ok=True)
with open(TRAIN_PATH, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)
with open(VAL_PATH, "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2, ensure_ascii=False)
print(f"Saved to {TRAIN_PATH} and {VAL_PATH}")

In [ ]:
# Preview tokenization
from transformers import AutoTokenizer

model_name = config["model_name"]
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
except Exception:
    model_name = config["fallback_model"]
    print(f"Fallback to: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokens = tokenizer(train_data[0]["text"], truncation=True, max_length=2048)
print(f"Tokenizer: {model_name} | Vocab: {tokenizer.vocab_size}")
print(f"Sample token count: {len(tokens['input_ids'])} | Max seq length: {config['max_seq_length']}")

token_lengths = [len(tokenizer(s["text"], truncation=True, max_length=2048)["input_ids"]) for s in train_data[:50]]
print(f"Token lengths (first 50) — Min: {min(token_lengths)}, Max: {max(token_lengths)}, Mean: {sum(token_lengths)/len(token_lengths):.0f}")
print("\nDataset preparation complete!")

---
## 3. Model Loading & LoRA Setup

In [ ]:
import yaml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

with open("configs/training_config.yaml", "r") as f:
    train_cfg = yaml.safe_load(f)
with open("configs/lora_config.yaml", "r") as f:
    lora_cfg = yaml.safe_load(f)

print("Training Config:")
print(f"  Model: {train_cfg['model_name']} | Fallback: {train_cfg['fallback_model']}")
print(f"  4-bit: {train_cfg['use_4bit']} | Max seq: {train_cfg['max_seq_length']}")
print(f"\nLoRA Config:")
print(f"  r={lora_cfg['r']}, alpha={lora_cfg['lora_alpha']}, dropout={lora_cfg['lora_dropout']}")
print(f"  targets: {lora_cfg['target_modules']}, bias: {lora_cfg['bias']}")

In [ ]:
# Quantization config
use_4bit = train_cfg["use_4bit"] and torch.cuda.is_available()
bnb_config = None
if use_4bit:
    compute_dtype = getattr(torch, train_cfg.get("bnb_4bit_compute_dtype", "float16"))
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=train_cfg.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=train_cfg.get("use_double_quant", True),
    )
    print("4-bit quantization configured (NF4 + double quant)")
else:
    print("Quantization disabled")

In [ ]:
# Load tokenizer
model_name = train_cfg["model_name"]
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Tokenizer loaded: {model_name}")
except Exception as e:
    print(f"Primary failed: {e}")
    model_name = train_cfg["fallback_model"]
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Fallback tokenizer: {model_name}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab: {tokenizer.vocab_size} | Pad: {tokenizer.pad_token} | EOS: {tokenizer.eos_token}")

In [ ]:
# Load base model
print(f"Loading base model: {model_name} ...")
model_kwargs = {"trust_remote_code": True, "torch_dtype": torch.float16, "device_map": "auto"}
if bnb_config is not None:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
print(f"Loaded: {type(model).__name__} | Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Apply LoRA adapters
if train_cfg.get("gradient_checkpointing", True):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

if bnb_config is not None:
    model = prepare_model_for_kbit_training(model)
    print("Prepared for k-bit training")

lora_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"\nTrainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Quick sanity check
test_input = tokenizer("### Instruction:\nEvaluate the resume", return_tensors="pt")
test_input = {k: v.to(model.device) for k, v in test_input.items()}
with torch.no_grad():
    output = model(**test_input)
print(f"Forward pass OK — logits shape: {output.logits.shape}")

---
## 4. Fine-Tuning

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Load processed datasets
with open(train_cfg["train_dataset"], "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    val_data = json.load(f)

assert "text" in train_data[0], "Run Section 2 first to prepare dataset with formatted prompts"
print(f"Train: {len(train_data)} | Validation: {len(val_data)}")

In [ ]:
# Tokenize datasets
max_seq_length = train_cfg.get("max_seq_length", 2048)

def tokenize_data(data, tokenizer, max_length):
    texts = [sample["text"] for sample in data]
    def tokenize_fn(examples):
        tokenized = tokenizer(examples["text"], truncation=True, max_length=max_length, padding="max_length")
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized
    dataset = Dataset.from_dict({"text": texts})
    return dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

print(f"Tokenizing (max_seq_length={max_seq_length})...")
train_dataset = tokenize_data(train_data, tokenizer, max_seq_length)
val_dataset = tokenize_data(val_data, tokenizer, max_seq_length)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Features: {train_dataset.features}")

In [ ]:
# Configure and launch training
output_dir = train_cfg["output_dir"]

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=train_cfg.get("num_train_epochs", 3),
    per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 2),
    per_device_eval_batch_size=train_cfg.get("per_device_eval_batch_size", 2),
    gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 8),
    learning_rate=train_cfg.get("learning_rate", 2e-4),
    warmup_steps=train_cfg.get("warmup_steps", 100),
    logging_steps=train_cfg.get("logging_steps", 10),
    eval_steps=train_cfg.get("eval_steps", 50),
    save_steps=train_cfg.get("save_steps", 100),
    optim=train_cfg.get("optim", "adamw_torch"),
    lr_scheduler_type=train_cfg.get("lr_scheduler_type", "linear"),
    weight_decay=train_cfg.get("weight_decay", 0.01),
    fp16=train_cfg.get("fp16", True),
    bf16=train_cfg.get("bf16", False),
    save_total_limit=train_cfg.get("save_total_limit", 3),
    load_best_model_at_end=train_cfg.get("load_best_model_at_end", True),
    eval_strategy=train_cfg.get("evaluation_strategy", "steps"),
    save_strategy=train_cfg.get("save_strategy", "steps"),
    logging_dir=train_cfg.get("logging_dir", f"{output_dir}/logs"),
    report_to=train_cfg.get("report_to", "none"),
    seed=train_cfg.get("seed", 42),
    gradient_checkpointing=train_cfg.get("gradient_checkpointing", True),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
print(f"Trainer ready — output: {output_dir}, epochs: {training_args.num_train_epochs}, "
      f"batch: {training_args.per_device_train_batch_size}, lr: {training_args.learning_rate}")

In [ ]:
# Run fine-tuning
print("Starting fine-tuning...")
print("=" * 50)

train_result = trainer.train()

print("\n" + "=" * 50)
print("Training complete!")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Save LoRA adapter and run final evaluation
print(f"Saving LoRA adapter to: {output_dir}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("\nSaved files:")
for f in sorted(Path(output_dir).glob("*")):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

print("\nRunning final evaluation...")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value}")

---
## 5. Inference Testing
The trained model is already in memory — switching to eval mode for inference.

In [ ]:
import re

model.eval()

def format_inference_prompt(instruction, resume_text, job_description):
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def extract_json(text):
    json_match = re.search(r'\{[\s\S]*\}', text)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict) or not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=1024, temperature=0.1):
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_inference_prompt(instruction, resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=0.9, repetition_penalty=1.1, do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}
    return result

print("Inference helpers ready")

In [ ]:
# Test Sample 1
sample_resume = """John Smith
Software Engineer | john.smith@email.com | (555) 123-4567

EXPERIENCE
Software Engineer, TechCorp Inc. - Jan 2021 - Present
- Developed RESTful APIs using Python and Flask serving 10K daily users
- Managed PostgreSQL databases with 50+ tables and optimized query performance
- Collaborated with cross-functional teams to deliver 3 major product releases
- Responsible for maintaining CI/CD pipelines using Jenkins and Docker

Junior Developer, StartupXYZ - Jun 2019 - Dec 2020
- Built frontend components using React and TypeScript
- Helped with bug fixes and code reviews
- Participated in daily standup meetings and sprint planning

EDUCATION
B.S. Computer Science, State University - 2019

SKILLS
Python, JavaScript, React, Flask, PostgreSQL, Docker, Git, AWS"""

sample_jd = """Senior Software Engineer
TechGlobal Inc.

Requirements:
- 5+ years of experience in software development
- Strong proficiency in Python and Java
- Experience with RESTful API design and microservices architecture
- Proficiency with SQL databases (PostgreSQL preferred)
- Experience with cloud services (AWS or GCP)
- Familiarity with Docker and Kubernetes
- Strong understanding of CI/CD pipelines

Responsibilities:
- Design and implement scalable backend services
- Write clean, maintainable, and well-tested code
- Participate in code reviews and technical design discussions
- Mentor junior engineers"""

print("Running inference on sample 1...")
result = generate_ats_eval(sample_resume, sample_jd)
print(json.dumps(result, indent=2))

In [ ]:
# Validate sample 1 output
if result.get("valid_json"):
    print(f"JSON Validation: PASSED | ATS Score: {result['ats_score']}/100")
    print(f"Matched Skills ({len(result['matched_skills'])}): {result['matched_skills']}")
    print(f"Missing Skills ({len(result['missing_skills'])}): {result['missing_skills']}")
    print(f"Weak Bullets: {len(result['weak_bullets'])} | Formatting Issues: {len(result['formatting_issues'])}")
    print(f"Feedback: {result['overall_feedback'][:200]}...")
else:
    print("JSON Validation: FAILED")
    print(f"Raw: {result.get('raw_output', 'N/A')[:500]}")

In [ ]:
# Test Sample 2
sample_resume_2 = """Sarah Chen
Data Scientist | sarah.chen@email.com

EXPERIENCE
Senior Data Scientist, DataDriven Corp - Mar 2022 - Present
- Built machine learning models achieving 92% accuracy on customer churn prediction
- Designed A/B testing framework that increased conversion rates by 15%
- Processed and analyzed datasets containing 10M+ records using PySpark

Data Analyst, Analytics Co - Aug 2020 - Feb 2022
- Created dashboards using Tableau
- Responsible for data cleaning tasks
- Assisted in building predictive models

EDUCATION
M.S. Statistics, Top University - 2020

SKILLS
Python, R, SQL, TensorFlow, PyTorch, Tableau, PySpark, scikit-learn"""

sample_jd_2 = """Machine Learning Engineer
AIVentures Inc.

Requirements:
- 3+ years of ML engineering experience
- Strong Python skills
- Experience with PyTorch or TensorFlow
- Experience deploying ML models to production
- Knowledge of MLOps (MLflow, Kubeflow)
- Experience with NLP or Computer Vision

Responsibilities:
- Design and implement ML pipelines
- Deploy and monitor models in production
- Optimize model performance"""

print("Running inference on sample 2...")
result_2 = generate_ats_eval(sample_resume_2, sample_jd_2)
print(json.dumps(result_2, indent=2))
if result_2.get('valid_json'):
    print(f"\nATS Score: {result_2['ats_score']}/100")

In [ ]:
# Determinism test
print("Testing output consistency (temperature=0.0)...")
result_det1 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)
result_det2 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)

if result_det1.get("valid_json") and result_det2.get("valid_json"):
    score_match = result_det1["ats_score"] == result_det2["ats_score"]
    skills_match = result_det1["matched_skills"] == result_det2["matched_skills"]
    print(f"Score match: {score_match} ({result_det1['ats_score']} vs {result_det2['ats_score']})")
    print(f"Skills match: {skills_match}")
    print(f"Deterministic: {'YES' if score_match and skills_match else 'PARTIAL'}")
else:
    print("Cannot test determinism - invalid JSON output")

---
## 6. Evaluation Metrics
Batch inference on the full validation set.

In [ ]:
from tqdm import tqdm

# Load validation data
with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    eval_val_data = json.load(f)
print(f"Validation samples: {len(eval_val_data)}")

def generate_single(sample, max_new_tokens=1024):
    prompt = f"### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.1,
            top_p=0.9, repetition_penalty=1.1, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
# Run batch inference
print(f"Running inference on {len(eval_val_data)} validation samples...\n")

results = []
json_valid_count = 0
ats_valid_count = 0
predicted_scores = []
ground_truth_scores = []
predicted_skills = []
gt_skills = []

for i, sample in enumerate(tqdm(eval_val_data)):
    raw_output = generate_single(sample)
    parsed = extract_json(raw_output)
    gt = json.loads(sample["output"])

    is_valid_json = parsed is not None
    is_valid_ats = is_valid_json and validate_ats_output(parsed)

    if is_valid_json:
        json_valid_count += 1
    if is_valid_ats:
        ats_valid_count += 1
        predicted_scores.append(parsed["ats_score"])
        ground_truth_scores.append(gt["ats_score"])
        predicted_skills.append(set(parsed.get("missing_skills", [])))
        gt_skills.append(set(gt.get("missing_skills", [])))

    results.append({"index": i, "valid_json": is_valid_json, "valid_ats": is_valid_ats,
                     "predicted": parsed, "ground_truth": gt})

print("\nInference complete!")

In [ ]:
# Evaluation metrics
total = len(eval_val_data)
json_rate = json_valid_count / total * 100
ats_rate = ats_valid_count / total * 100

print("=" * 50)
print("EVALUATION METRICS")
print("=" * 50)

print(f"\n1. JSON Validity Rate")
print(f"   Valid JSON: {json_valid_count}/{total} ({json_rate:.1f}%)")
print(f"   Valid ATS:  {ats_valid_count}/{total} ({ats_rate:.1f}%)")
print(f"   Target >95%: {'PASSED' if json_rate >= 95 else 'BELOW TARGET'}")

print(f"\n2. ATS Score Distribution")
if predicted_scores:
    print(f"   Predicted — Min: {min(predicted_scores)}, Max: {max(predicted_scores)}, Mean: {sum(predicted_scores)/len(predicted_scores):.1f}")
    print(f"   Ground truth — Min: {min(ground_truth_scores)}, Max: {max(ground_truth_scores)}, Mean: {sum(ground_truth_scores)/len(ground_truth_scores):.1f}")
    diffs = [abs(p - g) for p, g in zip(predicted_scores, ground_truth_scores)]
    mean_diff = sum(diffs) / len(diffs)
    print(f"   Mean absolute difference: {mean_diff:.1f}")

    buckets = Counter((s // 10) * 10 for s in predicted_scores)
    for bucket in sorted(buckets):
        print(f"     {bucket:3d}-{bucket+9:3d}: {'#' * buckets[bucket]} ({buckets[bucket]})")
else:
    print("   No valid predictions to analyze")

print(f"\n3. Missing Skill Accuracy")
if predicted_skills:
    total_overlap = total_gt = total_pred = 0
    for pred, gt in zip(predicted_skills, gt_skills):
        pred_lower = {s.lower() for s in pred}
        gt_lower = {s.lower() for s in gt}
        total_overlap += len(pred_lower & gt_lower)
        total_gt += len(gt_lower)
        total_pred += len(pred_lower)
    recall = total_overlap / total_gt * 100 if total_gt else 0
    precision = total_overlap / total_pred * 100 if total_pred else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    print(f"   Precision: {precision:.1f}% | Recall: {recall:.1f}% | F1: {f1:.1f}%")
else:
    print("   No valid predictions to analyze")

In [ ]:
# Sample predictions comparison
print("\n" + "=" * 50)
print("SAMPLE PREDICTIONS")
print("=" * 50)

for i, r in enumerate(results[:5]):
    gt = r["ground_truth"]
    pred = r["predicted"]
    print(f"\n--- Sample {i+1} --- JSON: {r['valid_json']} | ATS: {r['valid_ats']}")
    if r["valid_ats"] and pred:
        print(f"  GT Score: {gt['ats_score']} | Pred Score: {pred['ats_score']} | Diff: {abs(gt['ats_score'] - pred['ats_score'])}")
        print(f"  GT Missing:   {gt['missing_skills'][:3]}")
        print(f"  Pred Missing: {pred['missing_skills'][:3]}")
    else:
        print(f"  Prediction: {str(pred)[:200] if pred else 'None'}")

In [ ]:
# Final summary
print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"  Validation samples: {total}")
print(f"  JSON validity: {json_rate:.1f}%")
print(f"  ATS structure: {ats_rate:.1f}%")
if predicted_scores:
    print(f"  Mean score diff: {mean_diff:.1f}")
print(f"\n  Quality targets:")
print(f"    JSON validity >95%:  {'PASS' if json_rate > 95 else 'FAIL'}")
print(f"    Consistent scoring:  {'PASS' if predicted_scores and mean_diff < 20 else 'NEEDS IMPROVEMENT'}")
print(f"    Valid ATS >90%:      {'PASS' if ats_rate > 90 else 'FAIL'}")
print(f"\n  LoRA adapter saved to: {output_dir}")
print("\nPipeline complete!")